# 리뷰 데이터 로드 및 전처리

`data/raw/`에 저장된 리뷰 데이터를 로드하여 전처리한 뒤 `data/preprocessed/`에 저장한다.

## 로드 대상

| 파일 | 설명 |
|---|---|
| `steam_indie_reviews.csv` | 리뷰 원문 및 작성자 정보 |
| `steam_indie_review_summary.csv` | 게임별 리뷰 요약 통계 |
| `steam_indie_review_histogram.csv` | 월별/일별 리뷰 집계 |

## 전처리 결과

| 대상 | 처리 내용 | 저장 파일 |
|---|---|---|
| `steam_indie_review_summary` | `review_score_desc` 소문자 변환 | `data/preprocessed/steam_indie_review_summary.csv` |
| `steam_indie_review_histogram` | 파생 컬럼 추가, 분석 대상 필터링 | `data/preprocessed/steam_indie_review_histogram.csv` |
| `steam_indie_reviews` | 결측 제거, 날짜 변환, 플레이타임 정제 | `data/preprocessed/steam_indie_reviews.csv` |

## 라이브러리 임포트 및 데이터 로드

In [12]:
import warnings
import numpy as np
import pandas as pd
from pathlib import Path

warnings.filterwarnings('ignore')

# VS Code 노트북: __vsc_ipynb_file__ 로 프로젝트 루트 계산
if '__vsc_ipynb_file__' in globals():
    PROJECT_ROOT = Path(globals()['__vsc_ipynb_file__']).parents[2]
else:
    import subprocess
    _out = subprocess.run(
        ['git', 'rev-parse', '--show-toplevel'],
        capture_output=True, text=True, cwd=Path.home()
    )
    PROJECT_ROOT = Path(_out.stdout.strip())

RAW_DIR          = PROJECT_ROOT / 'data' / 'raw'
PREPROCESSED_DIR = PROJECT_ROOT / 'data' / 'preprocessed'
RAW_DIR.mkdir(parents=True, exist_ok=True)
PREPROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print(f'PROJECT_ROOT : {PROJECT_ROOT}')
print(f'RAW_DIR      : {RAW_DIR}')
print(f'PREPROCESSED : {PREPROCESSED_DIR}')

DB 연결 성공
RAW_DIR: /Users/jin/Develop/codingclub/game-analysis/data/raw
PREPROCESSED_DIR: /Users/jin/Develop/codingclub/game-analysis/data/preprocessed


## 1. steam_indie_reviews

수집된 전체 리뷰 데이터. 용량이 크므로 청크 단위로 읽어 저장한다.

In [13]:
out = RAW_DIR / 'steam_indie_reviews.csv'
if out.exists():
    print(f'steam_indie_reviews 파일 확인 완료: {out.resolve()}')
    print(f'파일 크기: {out.stat().st_size / 1024 / 1024:.1f} MB')
else:
    raise FileNotFoundError(f'{out} 파일이 없습니다. 데이터 수집 스크립트를 먼저 실행하세요.')

  청크 1: 50,000행 저장 완료
  청크 2: 100,000행 저장 완료
  청크 3: 150,000행 저장 완료
  청크 4: 161,459행 저장 완료

steam_indie_reviews: 총 161,459행 → steam_indie_reviews.csv


## 2. steam_indie_review_summary

게임별 리뷰 요약 통계 (review_score, 긍정/부정 수 등). 수집 시점의 전체 누적 리뷰 기준이다.

In [14]:
df_summary = pd.read_csv(RAW_DIR / 'steam_indie_review_summary.csv')

print(f'steam_indie_review_summary: {len(df_summary):,}행')
print(f'컬럼: {df_summary.columns.tolist()}')
df_summary.head(3)

steam_indie_review_summary: 200행 → steam_indie_review_summary.csv
컬럼: ['appid', 'review_score', 'review_score_desc', 'total_positive', 'total_negative', 'total_reviews']


,appid,review_score,review_score_desc,total_positive,total_negative,total_reviews
0,324470,6,Mostly Positive,118,33,151
1,571740,8,Very Positive,22066,2262,24328
2,588440,8,Very Positive,64,14,78


## 3. steam_indie_review_histogram

게임별 월별(`rollups`) 및 일별(`recent`) 리뷰 집계 데이터.

In [15]:
df_hist = pd.read_csv(RAW_DIR / 'steam_indie_review_histogram.csv')

print(f'steam_indie_review_histogram: {len(df_hist):,}행')
print(f'컬럼: {df_hist.columns.tolist()}')
df_hist.head(3)

steam_indie_review_histogram: 493,459행 → steam_indie_review_histogram.csv
컬럼: ['appid', 'name', 'release_date', 'hist_start_date', 'hist_end_date', 'date', 'recommendations_up', 'recommendations_down', 'data_type']

data_type 분포:
data_type
recent     267570
rollups    225889


,appid,name,release_date,hist_start_date,hist_end_date,date,recommendations_up,recommendations_down,data_type
0,226620,Desktop Dungeons,2023-04-18,2013-11-07,2026-04-25,2026-03-27,0,0,recent
1,226620,Desktop Dungeons,2023-04-18,2013-11-07,2026-04-25,2026-03-28,0,0,recent
2,226620,Desktop Dungeons,2023-04-18,2013-11-07,2026-04-25,2026-03-29,0,0,recent


## 데이터 로드 요약

In [16]:
print('=== 로드 완료 파일 목록 ===')
for f in sorted(RAW_DIR.glob('steam_indie_review*.csv')):
    size_mb = f.stat().st_size / 1024 / 1024
    print(f'  {f.name:<50} {size_mb:6.1f} MB')

DB 연결 종료

=== 저장 완료 파일 목록 ===
  steam_indie_review_histogram.csv                      38.6 MB
  steam_indie_review_summary.csv                         0.0 MB
  steam_indie_reviews.csv                               62.7 MB


---

## steam_indie_review_summary 전처리

- `review_score_desc`: 소문자 변환
- → `data/preprocessed/steam_indie_review_summary.csv`

In [17]:
df_summary['review_score_desc'] = df_summary['review_score_desc'].str.lower()

out_path = PREPROCESSED_DIR / 'steam_indie_review_summary.csv'
df_summary.to_csv(out_path, index=False)
print(f'저장 완료 → {out_path} ({len(df_summary):,}개)')
print(df_summary['review_score_desc'].value_counts().to_string())

저장 완료 → /Users/jin/Develop/codingclub/game-analysis/data/preprocessed/steam_indie_review_summary.csv (200개)
review_score_desc
very positive              71
positive                   49
mostly positive            26
mixed                      24
overwhelmingly positive     7
8 user reviews              4
9 user reviews              4
mostly negative             3
no user reviews             2
6 user reviews              2
4 user reviews              2
7 user reviews              2
5 user reviews              2
2 user reviews              1
negative                    1


---

## steam_indie_review_histogram 전처리

| 처리 항목 | 결과 |
|---|---|
| 문자열 공백 | 문자열 컬럼 앞뒤 공백 제거, `data_type` 소문자 통일 |
| 날짜 변환 | `release_date`, `hist_start_date`, `hist_end_date`, `date` → datetime |
| 리뷰 수 합계 | `recommendations_total` 생성 (`recommendations_up + recommendations_down`) |
| 분석 대상 필터링 | `steam_indie_games` 기준 inner join (분석 대상 게임의 히스토그램만 유지) |

In [18]:
df_hist = pd.read_csv(RAW_DIR / 'steam_indie_review_histogram.csv')
print(f'로드 완료: {df_hist.shape}')
print(f'컬럼: {df_hist.columns.tolist()}')

로드 완료: (493459, 9)
컬럼: ['appid', 'name', 'release_date', 'hist_start_date', 'hist_end_date', 'date', 'recommendations_up', 'recommendations_down', 'data_type']


In [19]:
# 문자열 공백 정리
string_cols = df_hist.select_dtypes(include=['object', 'string']).columns.tolist()
for col in string_cols:
    df_hist[col] = df_hist[col].astype('string').str.strip()
if 'name' in df_hist.columns:
    df_hist['name'] = df_hist['name'].str.replace(r'\s+', ' ', regex=True)
df_hist['data_type'] = df_hist['data_type'].str.lower()

# 날짜형 변환
for col in ['release_date', 'hist_start_date', 'hist_end_date', 'date']:
    df_hist[col] = pd.to_datetime(df_hist[col], errors='coerce')

# 리뷰 수 합계
for col in ['appid', 'recommendations_up', 'recommendations_down']:
    df_hist[col] = pd.to_numeric(df_hist[col], errors='coerce')
df_hist['recommendations_total'] = df_hist['recommendations_up'] + df_hist['recommendations_down']

# steam_indie_games 기준 inner join (분석 대상 게임의 히스토그램만 유지)
games_appids = pd.read_csv(PREPROCESSED_DIR / 'steam_indie_games.csv', usecols=['appid'])
before = len(df_hist)
df_hist = df_hist.merge(games_appids, on='appid', how='inner')
print(f'inner join 결과: {before:,} → {len(df_hist):,}행 ({before - len(df_hist):,}개 제외)')

print('전처리 완료')
print(f'shape: {df_hist.shape}')
print(f'컬럼: {df_hist.columns.tolist()}')

inner join 결과: 493,459 → 468,589행 (24,870개 제외)
전처리 완료
shape: (468589, 10)
컬럼: ['appid', 'name', 'release_date', 'hist_start_date', 'hist_end_date', 'date', 'recommendations_up', 'recommendations_down', 'data_type', 'recommendations_total']


In [20]:
out_path = PREPROCESSED_DIR / 'steam_indie_review_histogram.csv'
df_hist.to_csv(out_path, index=False)
print(f'저장 완료 → {out_path} ({len(df_hist):,}개)')

저장 완료 → /Users/jin/Develop/codingclub/game-analysis/data/preprocessed/steam_indie_review_histogram.csv (468,589개)


---

## steam_indie_reviews 전처리

| 처리 항목 | 결과 |
|---|---|
| 결측 제거 | `review` 컬럼 결측 행 제거 |
| 분석 대상 필터링 | `steam_indie_games` 기준 inner join (분석 대상 게임 리뷰만 유지) |
| 날짜 변환 | `timestamp_created` → `created_date`, `timestamp_updated` → `updated_date`, `author_last_played` → `author_last_played_date` |
| 플레이타임 | `author_playtime_*` 3개 컬럼 분 → 시간 단위 변환 후 원본 제거 |

In [21]:
reviews = pd.read_csv(RAW_DIR / 'steam_indie_reviews.csv')
print(f'로드 완료: {reviews.shape}')

# 결측 제거
reviews_clean = reviews.copy()
reviews_clean = reviews_clean.dropna(subset=['review'])

# steam_indie_games 기준 inner join (분석 대상 게임의 리뷰만 유지)
games_appids = pd.read_csv(PREPROCESSED_DIR / 'steam_indie_games.csv', usecols=['appid'])
before = len(reviews_clean)
reviews_clean = reviews_clean.merge(games_appids, on='appid', how='inner')
print(f'inner join 결과: {before:,} → {len(reviews_clean):,}행 ({before - len(reviews_clean):,}개 제외)')

# 날짜 변환 (Unix timestamp → datetime)
for col in ['timestamp_created', 'timestamp_updated', 'author_last_played']:
    new_col = col.replace('timestamp_', '') + '_date' if col.startswith('timestamp_') else 'author_last_played_date'
    reviews_clean[new_col] = pd.to_datetime(reviews_clean[col], unit='s', errors='coerce')

# 플레이타임 분 → 시간 변환 후 원본 제거
playtime_cols = ['author_playtime_forever', 'author_playtime_last_two_weeks', 'author_playtime_at_review']
for col in playtime_cols:
    new_col = col.replace('author_playtime_', 'playtime_') + '_hours'
    reviews_clean[new_col] = reviews_clean[col] / 60
reviews_clean = reviews_clean.drop(columns=playtime_cols)

print(f'전처리 완료: {reviews_clean.shape}')
print(f'컬럼: {reviews_clean.columns.tolist()}')
reviews_clean.head(5)

로드 완료: (161459, 21)
inner join 결과: 161,079 → 160,644행 (435개 제외)
전처리 완료: (160644, 24)
컬럼: ['recommendationid', 'appid', 'language', 'review', 'timestamp_created', 'timestamp_updated', 'voted_up', 'votes_up', 'votes_funny', 'weighted_vote_score', 'comment_count', 'steam_purchase', 'received_for_free', 'written_during_early_access', 'author_steamid', 'author_num_games_owned', 'author_num_reviews', 'author_last_played', 'created_date', 'updated_date', 'author_last_played_date', 'playtime_forever_hours', 'playtime_last_two_weeks_hours', 'playtime_at_review_hours']


,recommendationid,appid,language,review,timestamp_created,timestamp_updated,voted_up,votes_up,votes_funny,weighted_vote_score,...,author_steamid,author_num_games_owned,author_num_reviews,author_last_played,created_date,updated_date,author_last_played_date,playtime_forever_hours,playtime_last_two_weeks_hours,playtime_at_review_hours
0,18698790,324470,french,good game for this price,1445883033,1445883033,True,2,1,0.523810,...,76561197986244887,0,3,1503258537,2015-10-26 18:10:33,2015-10-26 18:10:33,2017-08-20 19:48:57,3.483333,0.0,1.250000
1,18699465,324470,english,"BGM - GOOD \nGRAPHICS - GOOD\n\nBut, Slip effe...",1445885616,1445885616,True,1,0,0.421372,...,76561198049920411,0,3,1445958836,2015-10-26 18:53:36,2015-10-26 18:53:36,2015-10-27 15:13:56,0.216667,0.0,0.216667
2,18699648,324470,english,"this game is like a zen-garden, I love it! \n\...",1445886217,1482541338,True,5,0,0.500076,...,76561198047893068,435,17,1623275518,2015-10-26 19:03:37,2016-12-24 01:02:18,2021-06-09 21:51:58,13.616667,0.0,12.666667
3,18700348,324470,english,Ever played Bhop? Surf? If so this games mecha...,1445889159,1445895001,True,16,0,0.637511,...,76561198045694190,0,3,1541537929,2015-10-26 19:52:39,2015-10-26 21:30:01,2018-11-06 20:58:49,7.483333,0.0,0.916667
4,18701774,324470,english,It's Lit,1445895144,1445895144,True,4,0,0.495810,...,76561198047336040,1631,56,1759549378,2015-10-26 21:32:24,2015-10-26 21:32:24,2025-10-04 03:42:58,8.666667,0.0,6.416667


In [22]:
out_path = PREPROCESSED_DIR / 'steam_indie_reviews.csv'
reviews_clean.to_csv(out_path, index=False)
print(f'저장 완료 → {out_path} ({len(reviews_clean):,}개)')

저장 완료 → /Users/jin/Develop/codingclub/game-analysis/data/preprocessed/steam_indie_reviews.csv (160,644개)
